In [1]:
import numpy as np
import pandas as pd


In [2]:
books=pd.read_csv('books.csv')
ratings=pd.read_csv('ratings.csv')
interactions=pd.read_csv('to_read.csv')

In [3]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9997 entries, 0 to 9996
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              9997 non-null   int64  
 1   book_id         9997 non-null   int64  
 2   description     9997 non-null   object 
 3   title           9997 non-null   object 
 4   language_code   8913 non-null   object 
 5   average_rating  9997 non-null   float64
 6   ratings_count   9997 non-null   int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 546.8+ KB


In [4]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 981756 entries, 0 to 981755
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   book_id  981756 non-null  int64
 1   user_id  981756 non-null  int64
 2   rating   981756 non-null  int64
dtypes: int64(3)
memory usage: 22.5 MB


In [5]:
interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912705 entries, 0 to 912704
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   user_id  912705 non-null  int64
 1   book_id  912705 non-null  int64
dtypes: int64(2)
memory usage: 13.9 MB


In [6]:
books.head()

,id,book_id,description,title,language_code,average_rating,ratings_count
0,1,2767052,Suzanne Collins The Hunger Games The Hunger Ga...,"The Hunger Games (The Hunger Games, #1)",eng,4.34,4780653
1,2,3,"J.K. Rowling, Mary GrandPré Harry Potter and t...",Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,4602479
2,3,41865,"Stephenie Meyer Twilight Twilight (Twilight, #1)","Twilight (Twilight, #1)",en-US,3.57,3866839
3,4,2657,Harper Lee To Kill a Mockingbird To Kill a Moc...,To Kill a Mockingbird,eng,4.25,3198671
4,5,4671,F. Scott Fitzgerald The Great Gatsby The Great...,The Great Gatsby,eng,3.89,2683664


In [7]:
ratings.head()

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


In [8]:
interactions.head()

,user_id,book_id
0,1,112
1,1,235
2,1,533
3,1,1198
4,1,1874


Cosine similarity....Description

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books['description'])

In [11]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [12]:
indices = pd.Series(books.index,index=books['title']).drop_duplicates()

In [13]:
def recommendations(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    book_indices = [i[0] for i in sim_scores]

    return books[['title', 'description']].iloc[book_indices]

Ratings....SVD

In [14]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

In [15]:
user_item_matrix = ratings.pivot_table(index='user_id',columns='book_id',values='rating').fillna(0)

In [16]:
matrix = csr_matrix(user_item_matrix.values)

In [17]:
svd = TruncatedSVD(
    n_components=20,
    random_state=42
)

matrix_svd = svd.fit_transform(matrix)

In [18]:
predicted_ratings = np.dot(
    matrix_svd,
    svd.components_
)

In [19]:
pred_df = pd.DataFrame(
    predicted_ratings,
    columns=user_item_matrix.columns,
    index=user_item_matrix.index
)

Combining Both

In [20]:
def collaborative_recommendation(user_id, top_n=10):

    if user_id not in pred_df.index:
        return "User not found"

    user_ratings = pred_df.loc[user_id]

    recommendations = user_ratings.sort_values(
        ascending=False
    ).head(top_n)

    recommended_books = books[
        books['book_id'].isin(recommendations.index)
    ]

    return recommended_books[
        ['book_id', 'title', "description", 'average_rating']
    ]

In [21]:
collaborative_recommendation(1)

,book_id,title,description,average_rating


In [22]:
def hybrid_recommendation(user_id, top_n=10):

    if user_id not in pred_df.index:
        return recommend_for_new_user()

    collab_scores = pred_df.loc[user_id]

    scores = []

    for book_id in collab_scores.index:

        collaborative_score = collab_scores[book_id]

        try:
            content_score = books[
                books['book_id'] == book_id
            ]['average_rating'].values[0]
        except:
            content_score = 0

        final_score = (
            0.7 * collaborative_score +
            0.3 * content_score
        )

        scores.append((book_id, final_score))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_books = scores[:top_n]

    recommended_ids = [
        x[0] for x in top_books
    ]

    return books[
        books['book_id'].isin(recommended_ids)
    ][[
        'book_id',
        'title',
        'average_rating'
    ]]

For New Users

In [23]:
def recommend_for_new_user(top_n=10):

    return books.sort_values(
        by='average_rating',
        ascending=False
    ).head(top_n)[[
        'book_id',
        'title',
        'average_rating'
    ]]

In [24]:
def recommend(user_id):

    if user_id not in ratings['user_id'].unique():
        return recommend_for_new_user()

    return hybrid_recommendation(user_id)

In [25]:
recommend(1)

,book_id,title,average_rating
17,5,Harry Potter and the Prisoner of Azkaban (Harr...,4.53
23,6,Harry Potter and the Goblet of Fire (Harry Pot...,4.53
26,1,Harry Potter and the Half-Blood Prince (Harry ...,4.54
892,870,"Fullmetal Alchemist, Vol. 1 (Fullmetal Alchemi...",4.49
963,30,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,4.59
3229,119,The Lord of the Rings: The Art of The Fellowsh...,4.59
3274,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",4.77
3752,10,"Harry Potter Collection (Harry Potter, #1-6)",4.73
4228,36,The Lord of the Rings: Weapons and Warfare,4.53
9135,5417,Carrie / 'Salem's Lot / The Shining,4.52


Evaluation

In [26]:
def precision_at_k(
    recommended,
    relevant,
    k=10
):

    recommended_k = recommended[:k]

    relevant_set = set(relevant)

    recommended_set = set(recommended_k)

    relevant_recommended = (
        recommended_set & relevant_set
    )

    return len(relevant_recommended) / k

In [27]:
def recall_at_k(
    recommended,
    relevant,
    k=10
):

    recommended_k = recommended[:k]

    relevant_set = set(relevant)

    recommended_set = set(recommended_k)

    relevant_recommended = (
        recommended_set & relevant_set
    )

    return (
        len(relevant_recommended)
        / len(relevant_set)
    )

In [30]:
from sklearn.metrics import ndcg_score

# choose a user
user_id = 1

# actual ratings by user
actual_ratings = ratings[
    ratings['user_id'] == user_id
]

# true relevance
true_relevance = actual_ratings['rating'].values

# predicted scores
predicted_scores = []

for book_id in actual_ratings['book_id']:

    predicted_scores.append(
        pred_df.loc[user_id, book_id]
    )

# reshape for sklearn
true_relevance = np.asarray([true_relevance])
predicted_scores = np.asarray([predicted_scores])

# compute ndcg
ndcg = ndcg_score(
    true_relevance,
    predicted_scores
)

print("NDCG:", ndcg)

NDCG: 0.9836821611850635
